[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C63_ML_System_Design_Course/04_serving_capacity/04_serving_capacity.ipynb)

# 04 · 服务、部署与容量估算（架构分层 / 锚点数字 / 容量心算 / 延迟预算 / 降级 / 监控）

目标：把「容量估算」从「凭感觉说个数」变成**几行可以运行、可以断言的心算工具**。

本 notebook 你会亲手实现：
1. **锚点数字表** —— 硬件算力/带宽、模型参数量/延迟、存储/带宽的量级速查（内置为数据结构）
2. **容量估算器** —— QPS→实例数、参数量→显存、FLOPs→算力、成本/千次推理，并做**两条独立路径互相校验**
3. **延迟预算分解器** —— 给一组子系统延迟，找出关键路径与瓶颈
4. **降级策略决策表** —— 给定触发信号，查表输出该降到哪一级
5. **监控告警阈值设定器** —— 给历史分布，按目标误报率反推告警阈值
6. **一个完整案例**：TSR 云端复检服务的端到端容量估算演示

> 心智模型：**每一个架构框图上的箭头，都要能配一个数量级数字；说不出数字的架构图只是一幅画。**

## 0 · 环境自检

本课全程只用标准库 + numpy。没有 GPU 依赖、不联网、不下载数据。

In [ ]:
import sys, math, random
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)

assert sys.version_info >= (3, 8), '需要 Python 3.8+'
assert hasattr(np, 'percentile')
print('\n✅ 环境自检通过：本课不需要 GPU、不需要联网。')

## 1 · 锚点数字表：把「量级」内置成数据结构

背这些表的目的不是精确，是**量级不错**（差一个数量级 = 错；差 2 倍 = 可接受）。

In [ ]:
# 硬件：峰值算力(TFLOPS, fp16/bf16) / 显存带宽(GB/s) / 典型显存(GB)
HW = {
    'T4':        dict(tflops=65,  bw_gb_s=320,  vram_gb=16),
    'A10_L4':    dict(tflops=140, bw_gb_s=450,  vram_gb=24),
    'A100':      dict(tflops=312, bw_gb_s=2000, vram_gb=80),
    'H100':      dict(tflops=990, bw_gb_s=3350, vram_gb=80),
    'car_soc':   dict(tflops=150, bw_gb_s=200,  vram_gb=12),   # 峰值；可用算力见下面的折扣
}
CAR_SOC_USABLE_FRACTION = 0.2   # 车端可用算力常只有峰值的 10-30%，取中间值 0.2 做心算

# 模型：参数量(M) / fp16 权重显存(MB) / 典型单请求延迟(ms, batch=1, 推理卡)
MODELS = {
    'tsr_tiny_det':   dict(params_m=6,    vram_mb=12,   latency_ms=6),
    'rtmdet_m':       dict(params_m=35,   vram_mb=70,   latency_ms=14),
    'resnet50_cls':   dict(params_m=25.6, vram_mb=51,   latency_ms=3),
    'vit_b16':        dict(params_m=86,   vram_mb=172,  latency_ms=10),
    'vla_7b':         dict(params_m=7000, vram_mb=14000, latency_ms=80),
}

# 带宽/存储量级
BANDWIDTH = dict(
    image_1080p_jpeg_mb=0.5,
    image_1080p_raw_mb=6.0,
    camera_fps=30,
    camera_count=6,
    cellular_uplink_mbps=10,     # 车云上行，保守估计
    datacenter_gbps=50,
)

assert HW['T4']['tflops'] < HW['A100']['tflops'] < HW['H100']['tflops']
assert MODELS['tsr_tiny_det']['params_m'] < MODELS['rtmdet_m']['params_m']
print('锚点数字表就位：', len(HW), '种硬件 ·', len(MODELS), '个模型量级 ·', len(BANDWIDTH), '条带宽/存储数字')
for name, d in HW.items():
    print(f'  {name:<10} {d["tflops"]:>6} TFLOPS  {d["bw_gb_s"]:>6} GB/s  {d["vram_gb"]:>3} GB')

## 2 · 容量估算器：QPS → 实例数

用 Little's Law（$L=\lambda W$）的变形：实例数 = ⌈QPS × 延迟(s) / 单实例并发度 × 余量⌉。

In [ ]:
def instances_needed(qps, latency_ms, concurrency_per_instance, margin=1.3):
    """QPS -> 所需实例数（向上取整，含余量系数）。"""
    latency_s = latency_ms / 1000.0
    raw = qps * latency_s / concurrency_per_instance
    return math.ceil(raw * margin)

# 案例：TSR 云端复检服务，QPS=200，单请求(batch=1)延迟10ms，
# SLA p99<50ms 下单卡批处理等效并发≈3.5
n = instances_needed(qps=200, latency_ms=10, concurrency_per_instance=3.5, margin=1.3)
assert n == 1, n   # 200*0.01/3.5*1.3 = 0.743 -> ceil = 1
n_hifi = instances_needed(qps=200, latency_ms=10, concurrency_per_instance=3.5, margin=1.3)
print(f'QPS=200, 延迟10ms, 并发3.5, 余量1.3x -> 至少 {n} 台（高可用通常再加 1 台冗余 -> 实际部署 2 台）')

# 反例：QPS 涨到 5000 时
n2 = instances_needed(qps=5000, latency_ms=10, concurrency_per_instance=3.5, margin=1.3)
assert n2 == 19, n2
print(f'QPS=5000 时 -> {n2} 台，说明流量涨 25 倍，实例数也约涨 25 倍（线性关系，符合 Little\'s Law 直觉）')
assert n2 / n >= 15   # 流量涨25倍，实例数量级也该跟着涨，而不是不变或超线性暴涨
print('\n✅ 容量估算器（QPS→实例数）就位。')

## 3 · 容量估算器：参数量 → 显存、FLOPs → 算力

In [ ]:
def vram_gb(params_m, bytes_per_param=2, activation_overhead_mb=200, framework_overhead_mb=500):
    """推理场景显存估算(GB)：权重 + 激活开销 + 框架固定开销。"""
    weight_mb = params_m * 1e6 * bytes_per_param / 1e6   # 参数量(M) * 1e6 -> 个数 * bytes / 1e6 -> MB
    total_mb = weight_mb + activation_overhead_mb + framework_overhead_mb
    return total_mb / 1024

v_tiny = vram_gb(MODELS['tsr_tiny_det']['params_m'])
v_vla = vram_gb(MODELS['vla_7b']['params_m'], activation_overhead_mb=2000, framework_overhead_mb=1000)
assert v_tiny < 1.0, v_tiny        # 几百 MB 量级
assert 13 < v_vla < 18, v_vla      # 7B fp16 权重约 14GB，加上开销略高
print(f'tsr_tiny_det 显存估算 ≈ {v_tiny:.2f} GB  (对应 notebook MODELS 表里的 vram_mb={MODELS["tsr_tiny_det"]["vram_mb"]}MB 权重量级一致)')
print(f'vla_7b       显存估算 ≈ {v_vla:.2f} GB  (单卡 24GB 推理卡装得下，但训练需要数倍显存)')

def tflops_needed(flops_per_infer, qps, utilization=0.3):
    """FLOPs -> 所需算力(TFLOPS)，按硬件利用率折扣。"""
    return flops_per_infer * qps / (utilization * 1e12)

t = tflops_needed(flops_per_infer=4e9, qps=1000, utilization=0.3)
assert 10 < t < 16, t
print(f'\n单帧4GFLOPs, QPS=1000, 利用率0.3 -> 需要算力 ≈ {t:.1f} TFLOPS')
print(f'  对比 T4 峰值 {HW["T4"]["tflops"]} TFLOPS -> 占用比例 {t/HW["T4"]["tflops"]*100:.0f}%，一张卡绰绰有余')
print('  结论：瓶颈大概率不在算力，而在延迟/调度开销 —— 这个判断本身就是加分点。')
print('\n✅ 容量估算器（显存/算力）就位。')

## 4 · 两条独立路径互相校验

数量级校验的核心方法：**用两条独立的估算路径算同一个量，量级应该吻合**。这是 C65-01 双路径校验的服务容量专项应用。

In [ ]:
def cross_check_instances(qps, latency_ms, single_gpu_flops, flops_per_infer, utilization=0.3):
    """路径A：按延迟/并发估算；路径B：按FLOPs/算力估算。两条路径应给出同一数量级的实例数。"""
    # 路径 A：显式并发度心算（假设 SLA 下单卡等效并发 3.5，与上面案例一致）
    n_a = instances_needed(qps, latency_ms, concurrency_per_instance=3.5, margin=1.3)
    # 路径 B：按算力总需求 / 单卡可用算力
    total_tflops_needed = flops_per_infer * qps / (utilization * 1e12)
    usable_tflops_per_gpu = single_gpu_flops * utilization
    n_b = math.ceil(total_tflops_needed / usable_tflops_per_gpu * 1.3)
    return n_a, n_b

na, nb = cross_check_instances(qps=200, latency_ms=10, single_gpu_flops=HW['T4']['tflops'],
                                 flops_per_infer=4e9, utilization=0.3)
print(f'路径A（延迟/并发） -> {na} 台')
print(f'路径B（FLOPs/算力）-> {nb} 台')
# 两条路径量级应该吻合（相差不超过一个数量级，理想情况下相差不超过2-3倍）
ratio = max(na, nb) / max(min(na, nb), 1)
assert ratio <= 5, f'两条路径结论相差过大({ratio:.1f}x)，说明某个估算假设有误，需要回头检查'
print(f'比值 {ratio:.1f}x，在可接受范围内 -> 两条路径互相印证，估算可信')
print('\n✅ 双路径校验就位：这是面试里「我用另一条路径验证一下」这句话的具体实现。')

## 5 · 延迟预算分解器：找关键路径与瓶颈

In [ ]:
def critical_path_analysis(stages, sla_ms):
    """stages: [(名字, 延迟ms, 是否在关键路径上), ...]
    返回 (关键路径总延迟, 剩余余量, 瓶颈stage名, 占比)"""
    on_path = [(name, ms) for name, ms, crit in stages if crit]
    total = sum(ms for _, ms in on_path)
    margin = sla_ms - total
    bottleneck = max(on_path, key=lambda x: x[1])
    share = bottleneck[1] / total
    return total, margin, bottleneck[0], share

STAGES = [
    ('预处理', 4, True), ('H2D拷贝', 2, True), ('模型推理', 12, True),
    ('后处理NMS', 3, True), ('跟踪关联', 2, True), ('序列化', 1, True),
    ('多帧滑窗累积(近线,并行)', 50, False),   # 不在关键路径上
    ('日志埋点(异步,并行)', 5, False),
]

total, margin, bn, share = critical_path_analysis(STAGES, sla_ms=100)
assert total == 24, total
assert margin == 76, margin
assert bn == '模型推理'
assert abs(share - 0.5) < 1e-9, share
print(f'关键路径合计 {total} ms，SLA 100ms 剩余余量 {margin} ms')
print(f'瓶颈子系统: {bn}（占关键路径 {share*100:.0f}%）')
print('并行分支（多帧滑窗、日志埋点）不计入端到端延迟，因为它们不在关键路径上。')

# 反直觉检验：如果把不在关键路径上的模块延迟"优化"为0，端到端延迟不应该变化
stages_optimized_wrong_target = [(n, 0 if n == '多帧滑窗累积(近线,并行)' else m, c) for n, m, c in STAGES]
total2, _, _, _ = critical_path_analysis(stages_optimized_wrong_target, sla_ms=100)
assert total2 == total, '优化非关键路径模块不应改变端到端关键路径延迟'
print('\n✅ 验证「假优化」陷阱：优化不在关键路径上的模块，端到端延迟纹丝不动。')

## 6 · 降级策略决策表

In [ ]:
DEGRADE_LEVELS = ['normal', 'model_degrade', 'feature_degrade', 'fallback_rule']

def degrade_decision(latency_p99_ms, sla_ms, consecutive_breaches, service_alive, lite_model_alive):
    """按触发条件查表返回应处于哪个降级层级。"""
    if not service_alive:
        return 'fallback_rule'
    if not lite_model_alive and latency_p99_ms > sla_ms:
        return 'feature_degrade'
    if latency_p99_ms > sla_ms and consecutive_breaches >= 3:
        return 'model_degrade'
    return 'normal'

cases = [
    dict(latency_p99_ms=30, sla_ms=100, consecutive_breaches=0, service_alive=True, lite_model_alive=True, want='normal'),
    dict(latency_p99_ms=120, sla_ms=100, consecutive_breaches=3, service_alive=True, lite_model_alive=True, want='model_degrade'),
    dict(latency_p99_ms=120, sla_ms=100, consecutive_breaches=1, service_alive=True, lite_model_alive=True, want='normal'),  # 未连续3次，不降级
    dict(latency_p99_ms=150, sla_ms=100, consecutive_breaches=5, service_alive=True, lite_model_alive=False, want='feature_degrade'),
    dict(latency_p99_ms=999, sla_ms=100, consecutive_breaches=10, service_alive=False, lite_model_alive=False, want='fallback_rule'),
]
for c in cases:
    want = c.pop('want')
    got = degrade_decision(**c)
    assert got == want, (c, got, want)
    print(f'{c} -> {got}')
print('\n✅ 降级决策表就位：触发条件 -> 降级层级是确定性映射，不是临场发挥。')

## 7 · 监控告警阈值设定器：按目标误报率反推阈值

In [ ]:
def alert_threshold(history, target_fpr=0.01):
    """history: 系统正常时的历史观测值数组。
    返回使得 P(观测值 > 阈值 | 正常) <= target_fpr 的阈值 —— 即 (1-target_fpr) 分位数。"""
    return float(np.percentile(history, (1 - target_fpr) * 100))

rng = np.random.default_rng(0)
# 模拟正常状态下的 p99 延迟观测（均值30ms，带一些抖动尾巴）
normal_latency = rng.gamma(shape=3.0, scale=10.0, size=10000)

thr_1pct = alert_threshold(normal_latency, target_fpr=0.01)
thr_01pct = alert_threshold(normal_latency, target_fpr=0.001)
assert thr_01pct > thr_1pct, '误报率要求越严格，阈值应该越高（更不容易触发）'

# 验证：用这个阈值反过来算，正常数据里超过阈值的比例应该约等于目标误报率
fpr_actual = float((normal_latency > thr_1pct).mean())
assert abs(fpr_actual - 0.01) < 0.01, fpr_actual   # 允许采样误差
print(f'目标误报率 1%   -> 阈值 {thr_1pct:.1f} ms  (实测误报率 {fpr_actual*100:.2f}%)')
print(f'目标误报率 0.1% -> 阈值 {thr_01pct:.1f} ms')
print('\n✅ 阈值设定器就位：用历史分位数反推阈值，而不是拍脑袋定一个绝对数字。')

## ✏️ 练习 1：容量估算的成本账

实现 `cost_per_thousand(gpu_hourly_cost, throughput_per_sec)`，返回每千次推理的成本（元）。

公式：每小时能处理 `throughput_per_sec * 3600` 次；每千次成本 = `gpu_hourly_cost / (throughput_per_sec*3600) * 1000`。

In [ ]:
def cost_per_thousand(gpu_hourly_cost, throughput_per_sec):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
c1 = cost_per_thousand(gpu_hourly_cost=8.0, throughput_per_sec=100)
assert abs(c1 - 8.0/360000*1000) < 1e-9, c1
assert 0.02 < c1 < 0.03, c1     # 量级：几分钱/千次
c2 = cost_per_thousand(gpu_hourly_cost=8.0, throughput_per_sec=10)
assert c2 == c1 * 10           # 吞吐降到1/10，单位成本涨10倍
print(f'吞吐100/s -> {c1:.4f} 元/千次')
print(f'吞吐10/s  -> {c2:.4f} 元/千次  (吞吐降到1/10，成本涨10倍，符合线性直觉)')
print('\n✅ 练习 1 通过：成本对比的价值在于横向比较不同方案，而不是精确到分。')

## ✏️ 练习 2：延迟预算的关键路径校验器

实现 `sla_check(stages, sla_ms)`，返回 `(是否达标, 关键路径延迟, 余量)`。
`stages` 格式同第 5 节 `critical_path_analysis`。达标条件：关键路径延迟 <= sla_ms。

In [ ]:
def sla_check(stages, sla_ms):
    # TODO: 复用 critical_path_analysis 的思路（只算 crit=True 的部分）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
ok, total, margin = sla_check(STAGES, sla_ms=100)
assert ok is True and total == 24 and margin == 76, (ok, total, margin)
ok2, total2, margin2 = sla_check(STAGES, sla_ms=20)
assert ok2 is False and total2 == 24 and margin2 == -4, (ok2, total2, margin2)
print(f'SLA=100ms -> 达标={ok}, 关键路径={total}ms, 余量={margin}ms')
print(f'SLA=20ms  -> 达标={ok2}, 关键路径={total2}ms, 余量={margin2}ms (负余量说明超标)')
print('\n✅ 练习 2 通过。')

## ✏️ 练习 3：车端可用算力折扣计算器

实现 `usable_tflops(hw_name, usable_fraction=None)`：返回 `HW[hw_name]['tflops'] * fraction`。
如果 `usable_fraction` 为 `None`，车端（`hw_name=='car_soc'`）用 `CAR_SOC_USABLE_FRACTION`，
其余硬件默认按 `0.35`（推理服务常见实测利用率）。

In [ ]:
def usable_tflops(hw_name, usable_fraction=None):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
u_car = usable_tflops('car_soc')
assert abs(u_car - HW['car_soc']['tflops'] * 0.2) < 1e-9, u_car
u_t4 = usable_tflops('T4')
assert abs(u_t4 - HW['T4']['tflops'] * 0.35) < 1e-9, u_t4
u_custom = usable_tflops('T4', usable_fraction=0.5)
assert abs(u_custom - HW['T4']['tflops'] * 0.5) < 1e-9, u_custom
print(f'car_soc 可用算力 ≈ {u_car:.1f} TFLOPS（峰值 {HW["car_soc"]["tflops"]} 的 20%）')
print(f'T4      可用算力 ≈ {u_t4:.1f} TFLOPS（峰值 {HW["T4"]["tflops"]} 的 35%）')
print('\n✅ 练习 3 通过：车端峰值看似接近云端小卡，但可用算力折扣完全不同，千万别直接比峰值。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def cost_per_thousand(gpu_hourly_cost, throughput_per_sec):
    per_hour_requests = throughput_per_sec * 3600
    return gpu_hourly_cost / per_hour_requests * 1000

In [ ]:
# 练习 2 参考答案
def sla_check(stages, sla_ms):
    total, margin, _, _ = critical_path_analysis(stages, sla_ms)
    return total <= sla_ms, total, margin

In [ ]:
# 练习 3 参考答案
def usable_tflops(hw_name, usable_fraction=None):
    if usable_fraction is None:
        usable_fraction = CAR_SOC_USABLE_FRACTION if hw_name == 'car_soc' else 0.35
    return HW[hw_name]['tflops'] * usable_fraction

---
## 🧪 真实工程胶囊：一份完整的容量估算演练模板（TSR 云端复检服务）

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# TSR 云端复检服务 —— 完整容量估算演练模板（可直接改数字复用）
# ══════════════════════════════════════════════════════════════════════

# 【第一步：写下已知条件】
# 目标 QPS: 200（车队规模 × 触发率折算而来，见 C58-03）
# 单请求延迟(batch=1): 10ms（rtmdet_m 级别模型）
# SLA: p99 < 50ms
# 模型参数量: 35M -> fp16 权重 70MB

# 【第二步：显存估算】
# 显存 ≈ 70MB(权重) + 200MB(激活,视输入分辨率) + 500MB(框架开销) ≈ 0.75 GB
# 结论：单卡 16GB 显存可同时装载 ~20 个这样的模型实例（显存不是瓶颈）

# 【第三步：实例数估算（路径A：延迟/并发）】
# 50ms SLA 下单卡批处理等效并发 ≈ 3.5
# 实例数 = ceil(200 * 0.01 / 3.5 * 1.3) = 1 台 -> 高可用部署 2 台

# 【第四步：算力估算（路径B：FLOPs/利用率，交叉校验）】
# 单帧 FLOPs ≈ 4e9, QPS=200, 利用率0.3
# 所需算力 ≈ 4e9*200/(0.3*1e12) ≈ 2.7 TFLOPS，远小于 T4 的 65 TFLOPS
# 两条路径都指向"1-2台T4即可" —— 互相印证

# 【第五步：成本估算】
# T4 云端时价约 8 元/小时，单卡吞吐 ~100 req/s -> 成本 ≈ 0.022 元/千次
# 200 QPS * 3600s = 72万次/小时 -> 每小时成本 ≈ 16 元（2台）

# 【第六步：延迟预算分解 + 关键路径】
# 预处理4 + 拷贝2 + 推理12 + 后处理3 + 跟踪2 + 序列化1 = 24ms 关键路径
# SLA 50ms，余量 26ms —— 留给排队延迟与偶发抖动

# 【第七步：降级方案】
# 触发：p99连续3次超50ms -> 切轻量模型(tsr_tiny_det, 6ms)
# 再触发：轻量模型也不可用 -> 跳过近线多帧融合，单帧直出
# 再触发：服务整体不可用 -> 维持上一个已知安全状态 + 告警

# 【第八步：监控】
# 数据漂移：亮度均值/目标尺寸分布 P1/P99 越界告警
# 指标漂移：分类别置信度滑窗均值，误报率控制在1%（连续3次超阈值才触发，抑制抖动）
# 延迟分布：p99硬阈值 + 分区数监控（防静默回退，见C60-05）
'''
print(RECIPE)
for token in ['QPS', 'SLA', 'TFLOPS', '路径A', '路径B', '降级方案', '监控', 'C58-03', 'C60-05']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：显存/实例数/算力/成本/延迟分解/降级/监控 —— 完整八步')

### 小结

- **服务与部署这一步，考的是「你的方案值多少台机器、多少钱、多快、坏了怎么办」**——
  能不能当场心算出数量级，是区分「纸上谈兵」与「能上线」的分水岭。
- **架构图要同时画数据流与控制流，并按在线/近线/离线三层分工**：
  等不了的放在线，能异步的放近线，能攒批的放离线；划分依据是 SLA，不是直觉。
- **锚点数字表要背，但记的是数量级不是小数点**：T4 几十 T 算力/几百 GB/s 带宽、
  A100 三百多 T/2TB 带宽是训练侧锚点；车端峰值看着高，**可用算力常只有峰值的 1-3 成**。
- **容量估算的四条心算公式**（QPS→实例数、参数量→显存、FLOPs→算力、成本/千次）
  背后是同一个 Little's Law（$L=\lambda W$），**永远用两条独立路径互相校验**。
- **延迟预算要找关键路径而不是简单相加**：只有串行、阻塞的部分才计入端到端延迟；
  优化不在关键路径上的模块是最常见的「假优化」。
- **降级方案不是可选项**：模型降级/特征降级/兜底规则要形成单调保守的层级，
  监控要覆盖数据漂移/指标漂移/延迟分布三类，且**关键类别必须独立监控，不能只看聚合指标**。

下一站：**模块 05 · 案例库：六个完整设计演练** ——
把七步框架 + 需求澄清 + 数据设计 + 建模评测 + 本模块的容量估算，
串成六个可以照着走一遍的完整 45 分钟演练脚本。